# 🎯 InsightFace (ArcFace 512-D) + LFW Genuine Photo Pairs + FAISS Vector Search
### Real Face Embedding Pipeline for Multi-Modal Suspect Ranking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-deepmind/antigravity/blob/main/insightface_faiss_colab.ipynb)

This notebook extracts genuine **512-dimensional ArcFace face embeddings** from real photograph pairs in the **Labeled Faces in the Wild (LFW)** dataset using **InsightFace** (`buffalo_l`).

**Key Methodological Rigor:**
- **Genuine Photo Pairs:** For each subject, **Photo 1** is registered in the gallery, and a **different, real Photo 2** of the same person (different angle, expression, lighting) serves as the query probe. No artificial Gaussian noise is added to the probe vector.
- **Anonymous Identifiers:** All individuals are strictly anonymized (`Subject_0001`, `Subject_0002`, etc.).
- **Contextual Attributes Disclosure:** While face vectors are genuine ArcFace embeddings from real photographs, non-facial contextual attributes (hair, build, age, zone, MO) are simulated to emulate investigative records.
- **FAISS Indexing:** Gallery embeddings are indexed in `faiss.IndexFlatIP(512)`.

## Step 1: Install Dependencies (GPU Accelerated)

In [ ]:
!pip install -q insightface onnxruntime-gpu faiss-gpu scikit-learn opencv-python matplotlib tqdm

## Step 2: Download LFW Dataset & Initialize InsightFace ArcFace (`buffalo_l`)

In [ ]:
import os, json, time
import numpy as np
import cv2
import faiss
import insightface
from insightface.app import FaceAnalysis
from sklearn.datasets import fetch_lfw_people
from tqdm.auto import tqdm

print('Initializing InsightFace buffalo_l on GPU/CUDA...')
app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))
print('InsightFace model ready!')

# Fetch subjects with at least 2 photos (Photo 1 for gallery, Photo 2 for probe)
print('Fetching LFW dataset via scikit-learn...')
lfw = fetch_lfw_people(min_faces_per_person=2, resize=1.0, color=True)
print(f'LFW loaded: {len(lfw.images)} photos across {len(lfw.target_names)} subjects.')

## Step 3: Extract Genuine ArcFace Embeddings on Photo Pairs
We map Photo 1 to the gallery and Photo 2 to the probe query. No Gaussian noise is added.

In [ ]:
def unit(x):
    norm = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.maximum(norm, 1e-12)

gallery_embeddings = []
probe_pairs = [] # (gallery_index, genuine_probe_vector)
target_to_gallery_idx = {}

for i, (img, target) in enumerate(tqdm(zip(lfw.images, lfw.target), total=len(lfw.images), desc='Extracting ArcFace Features')):
    img_bgr = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
    faces = app.get(img_bgr)
    if len(faces) == 0:
        continue
    emb = unit(faces[0].embedding.astype(np.float32))
    
    if target not in target_to_gallery_idx:
        # Photo 1 -> Registered in Gallery
        target_to_gallery_idx[target] = len(gallery_embeddings)
        gallery_embeddings.append(emb)
    else:
        # Photo 2 (different real photo of the same person) -> Query Probe
        g_idx = target_to_gallery_idx[target]
        probe_pairs.append((g_idx, emb))

gallery_embeddings = np.array(gallery_embeddings, dtype=np.float32)
n_gallery = len(gallery_embeddings)
print(f'Gallery size: {n_gallery} unique identities, dim={gallery_embeddings.shape[1]}')
print(f'Genuine probe cases from second photos: {len(probe_pairs)}')


## Step 4: Build FAISS Vector Index
We build a `faiss.IndexFlatIP(512)` for exact cosine inner-product search on the gallery.

In [ ]:
index = faiss.IndexFlatIP(512)
index.add(gallery_embeddings)
print(f'FAISS Index FlatIP built with {index.ntotal} vectors.')

# Quick sanity search on first 5 genuine probes
sample_probes = np.array([p[1] for p in probe_pairs[:5]], dtype=np.float32)
distances, indices = index.search(sample_probes, 5)
for i, (g_idx, _) in enumerate(probe_pairs[:5]):
    found_rank = list(indices[i]).index(g_idx) + 1 if g_idx in indices[i] else '>5'
    print(f'Probe {i}: True Subject {g_idx} found at FAISS rank #{found_rank} (Cosine sim: {distances[i][0]:.3f})')

## Step 5: Contextual Multi-Modal Attributes (Simulated)
*Note: Contextual attributes (hair, build, age, zone, MO) are simulated to emulate investigative records.*

In [ ]:
rng = np.random.default_rng(42)
ZONES, MO_TAGS = 30, 8
GROUP_NOISE = [1.0, 1.3, 1.8]

pop_hair = rng.integers(5, size=n_gallery)
pop_build = rng.integers(3, size=n_gallery)
pop_age = rng.integers(4, size=n_gallery)
pop_zones = rng.random((n_gallery, ZONES)) < 0.12
pop_mo = rng.integers(MO_TAGS, size=n_gallery)
pop_group = rng.choice(3, size=n_gallery, p=[0.6, 0.3, 0.1])
anonymous_names = np.array([f'Subject_{i:04d}' for i in range(n_gallery)])

pop = {
    'face': gallery_embeddings,
    'hair': pop_hair,
    'build': pop_build,
    'age': pop_age,
    'zones': pop_zones,
    'mo': pop_mo,
    'group': pop_group,
    'names': anonymous_names
}

# Build realistic surveillance probe cases
cases = []
for g_idx, genuine_probe_emb in probe_pairs:
    quality = float(rng.uniform(0.4, 1.0))
    def witness(true_val, k):
        if rng.random() > 0.7: return None
        return int(true_val) if rng.random() < 0.8 else int(rng.integers(k))
    zs = np.flatnonzero(pop_zones[g_idx])
    zone = int(rng.choice(zs)) if len(zs) and rng.random() < 0.8 else int(rng.integers(ZONES))
    cases.append({
        'truth': g_idx,
        'quality': quality,
        'probe': genuine_probe_emb,
        'zone': zone,
        'hair': witness(pop_hair[g_idx], 5),
        'build': witness(pop_build[g_idx], 3),
        'age': witness(pop_age[g_idx], 4),
        'mo': witness(pop_mo[g_idx], MO_TAGS)
    })

print(f'Synthesized {len(cases)} multi-modal cases from genuine LFW photo pairs.')

## Step 6: Feature Extraction & Learned Ranker Training

In [ ]:
FEATS = ['face', 'face_x_quality', 'hair', 'build', 'age', 'zone', 'mo']

def extract_features(pop, c):
    face = pop['face'] @ c['probe']
    def m(k):
        if c[k] is None: return np.zeros(len(face), dtype=np.float32)
        return np.where(pop[k] == c[k], 1.0, -1.0).astype(np.float32)
    return np.stack([
        face,
        face * c['quality'],
        m('hair'),
        m('build'),
        m('age'),
        pop['zones'][:, c['zone']].astype(np.float32),
        m('mo')
    ], axis=1).astype(np.float32)

train_cases = cases[:len(cases)//2]
test_cases = cases[len(cases)//2:]

print('Training Learned Ranker on held-out cases...')
X = [extract_features(pop, c) for c in train_cases]
truth = [c['truth'] for c in train_cases]
w = np.zeros(7, dtype=np.float32)
m_vec, v_vec = np.zeros_like(w), np.zeros_like(w)

for t in range(1, 301):
    g, loss = np.zeros_like(w), 0.0
    for Xc, y in zip(X, truth):
        s = Xc @ w
        p = np.exp(s - s.max())
        p /= p.sum()
        g += Xc.T @ p - Xc[y]
        loss -= np.log(p[y] + 1e-12)
    g = g / len(X) + 1e-3 * w
    m_vec = 0.9 * m_vec + 0.1 * g
    v_vec = 0.999 * v_vec + 0.001 * g * g
    w -= 0.05 * (m_vec / (1 - 0.9**t)) / (np.sqrt(v_vec / (1 - 0.999**t)) + 1e-8)
    if t % 50 == 0:
        print(f'  iter {t:3d} | train loss {loss / len(X):.3f}')

learned_weights = {f: round(float(val), 2) for f, val in zip(FEATS, w)}
print('\nLearned Feature Weights:', learned_weights)

## Step 7: Benchmark Evaluation on Genuine LFW Photo Pairs

In [ ]:
import pandas as pd

def evaluate_method(score_fn, ks=(1, 5, 10, 20)):
    ranks, truth_scores, all_scores, groups = [], [], [], []
    for c in test_cases:
        s = score_fn(c)
        t = s[c['truth']]
        ranks.append(1 + int((s > t).sum()))
        truth_scores.append(t)
        all_scores.append(s)
        groups.append(pop['group'][c['truth']])
    ranks, groups = np.array(ranks), np.array(groups)
    thr = np.percentile(truth_scores, 10)
    
    fprs = [(s[np.arange(len(s)) != c['truth']] >= thr).mean() for c, s in zip(test_cases, all_scores)]
    overall_fpr = float(np.mean(fprs))
    
    res = {f'Recall@{k}': float((ranks <= k).mean()) for k in ks}
    res['FPR@90%Recall'] = overall_fpr
    res['Analyst Time Saved'] = float(1 - ranks.mean() / (n_gallery / 2))
    return res

W_hand = {'face': 8.0, 'attr': 1.0, 'zone': 1.5, 'mo': 1.0}
def fn_face_only(c): return pop['face'] @ c['probe']
def fn_fused_hand(c):
    face = pop['face'] @ c['probe']
    obs = [(k, c[k]) for k in ('hair', 'build', 'age') if c[k] is not None]
    a = np.zeros(len(face))
    for k, v in obs: a += np.where(pop[k] == v, 1.0, -0.5)
    attr = W_hand['attr'] * a / max(len(obs), 1)
    zone = W_hand['zone'] * pop['zones'][:, c['zone']].astype(float)
    mo = W_hand['mo'] * (np.zeros(len(face)) if c['mo'] is None else (pop['mo'] == c['mo']).astype(float))
    return W_hand['face'] * c['quality'] * face + attr + zone + mo
def fn_learned(c): return extract_features(pop, c) @ w

df_results = pd.DataFrame({
    'Face-Only (FAISS)': evaluate_method(fn_face_only),
    'Fused Hand-Set': evaluate_method(fn_fused_hand),
    'Learned Ranker': evaluate_method(fn_learned)
})
print('\n=== BENCHMARK ON GENUINE LFW PHOTO PAIRS ===')
display(df_results.round(3))


## Step 8: Export `real_population.npz` and `weights.json` for Local Streamlit App

In [ ]:
from google.colab import files

np.savez_compressed(
    'real_population.npz',
    face=gallery_embeddings,
    hair=pop_hair,
    build=pop_build,
    age=pop_age,
    zones=pop_zones,
    mo=pop_mo,
    group=pop_group,
    names=anonymous_names
)

with open('weights.json', 'w') as f:
    json.dump(learned_weights, f, indent=2)

print('Files real_population.npz and weights.json created!')
files.download('real_population.npz')
files.download('weights.json')